# Risk Score Regression Model
This notebook builds a regression pipeline to predict risk scores with SMOTE balancing and feature engineering.

In [ ]:
# Import necessary libraries
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Add parent directory to path
sys.path.append(str(Path.cwd().parent))

# Import custom modules
from App_files.dataframe_loader import df_loader
from App_files.feature_engineering import feature_engineered
from App_files.feature_extraction import (
    correlation_Feature_selection,
    variance_Feature_selection,
    importance_feature_selection
)

## Step 1: Load DataFrame

In [ ]:
# Load the dataframe
df = df_loader()
print(f"DataFrame loaded with shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
df.head()

## Step 2: Check Initial Class Distribution

In [ ]:
# Check class distribution before SMOTE
if 'LoanApproved' in df.columns:
    print("Class distribution before SMOTE:")
    print(df['LoanApproved'].value_counts())
    print(f"\nClass proportions:")
    print(df['LoanApproved'].value_counts(normalize=True))
    
    # Visualize class distribution
    plt.figure(figsize=(8, 5))
    df['LoanApproved'].value_counts().plot(kind='bar')
    plt.title('Class Distribution Before SMOTE')
    plt.xlabel('Loan Approved')
    plt.ylabel('Count')
    plt.xticks(rotation=0)
    plt.show()
else:
    print("Warning: 'LoanApproved' column not found in dataframe")
    print(f"Available columns: {df.columns.tolist()}")

## Step 3: Apply Feature Engineering

In [ ]:
# Apply feature engineering
df_engineered = feature_engineered(df)
print(f"Shape after feature engineering: {df_engineered.shape}")
print(f"\nNew features added:")
new_features = set(df_engineered.columns) - set(df.columns)
print(new_features)
df_engineered.head()

## Step 4: Apply SMOTE for Class Balancing

In [ ]:
# Prepare data for SMOTE
# Separate features and target
X = df_engineered.drop('LoanApproved', axis=1)
y = df_engineered['LoanApproved']

# Select only numeric features for SMOTE
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
X_numeric = X[numeric_features]

print(f"Features before SMOTE: {X_numeric.shape}")
print(f"Target distribution before SMOTE:")
print(y.value_counts())

# Apply SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_numeric, y)

print(f"\nFeatures after SMOTE: {X_resampled.shape}")
print(f"\nTarget distribution after SMOTE:")
print(pd.Series(y_resampled).value_counts())
print(f"\nClass proportions after SMOTE:")
print(pd.Series(y_resampled).value_counts(normalize=True))

In [ ]:
# Visualize class distribution after SMOTE
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
y.value_counts().plot(kind='bar')
plt.title('Class Distribution Before SMOTE')
plt.xlabel('Loan Approved')
plt.ylabel('Count')
plt.xticks(rotation=0)

plt.subplot(1, 2, 2)
pd.Series(y_resampled).value_counts().plot(kind='bar')
plt.title('Class Distribution After SMOTE')
plt.xlabel('Loan Approved')
plt.ylabel('Count')
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

## Step 5: Feature Extraction/Selection

In [ ]:
# Create a dataframe from resampled data
df_resampled = pd.DataFrame(X_resampled, columns=numeric_features)
df_resampled['LoanApproved'] = y_resampled

# Apply correlation-based feature selection
selected_corr, dropped_corr = correlation_Feature_selection(df_resampled, threshold=0.85)
print(f"Correlation-based feature selection:")
print(f"Selected features: {len(selected_corr)}")
print(f"Dropped features: {dropped_corr}")

# Apply variance-based feature selection
selected_var, dropped_var = variance_Feature_selection(df_resampled, variance_threshold=0.0)
print(f"\nVariance-based feature selection:")
print(f"Selected features: {len(selected_var)}")
print(f"Dropped features: {dropped_var}")

# Get intersection of selected features
final_selected_features = list(set(selected_corr) & set(selected_var))
print(f"\nFinal selected features after both methods: {len(final_selected_features)}")
print(final_selected_features)

In [ ]:
# Apply importance-based feature selection
X_selected = df_resampled[final_selected_features]
y_selected = df_resampled['LoanApproved']

feature_importance, top_features = importance_feature_selection(X_selected, y_selected, top_k=10)
print(f"\nTop 10 important features:")
print(feature_importance.sort_values(ascending=False).head(10))

# Visualize feature importance
plt.figure(figsize=(10, 6))
feature_importance.sort_values(ascending=False).head(15).plot(kind='barh')
plt.title('Top 15 Feature Importances')
plt.xlabel('Importance')
plt.ylabel('Features')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Step 6: Build Regression Pipeline for Risk Score Prediction

In [ ]:
# Prepare data for regression
# Check if RiskScore exists or needs to be created
if 'RiskScore' in df_resampled.columns:
    print("RiskScore column found in data")
    target_col = 'RiskScore'
elif 'CreditScore' in df_resampled.columns:
    print("Using CreditScore as target for regression")
    target_col = 'CreditScore'
else:
    # Create a synthetic risk score based on available features
    print("Creating synthetic RiskScore based on available features")
    df_resampled['RiskScore'] = 100 - (df_resampled['LoanApproved'] * 50)
    target_col = 'RiskScore'
    
print(f"\nTarget column for regression: {target_col}")
print(f"Target statistics:")
print(df_resampled[target_col].describe())

In [ ]:
# Prepare final feature set for regression
# Exclude target column from features
feature_cols = [col for col in final_selected_features if col != target_col and col != 'LoanApproved']

X_regression = df_resampled[feature_cols]
y_regression = df_resampled[target_col]

print(f"Features for regression: {len(feature_cols)}")
print(f"Feature columns: {feature_cols}")
print(f"\nX shape: {X_regression.shape}")
print(f"y shape: {y_regression.shape}")

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_regression, y_regression, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

In [ ]:
# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features scaled successfully")

## Step 7: Train Multiple Regression Models

In [ ]:
# Define regression models
models = {
    'Ridge': Ridge(alpha=1.0, random_state=42),
    'Lasso': Lasso(alpha=1.0, random_state=42),
    'ElasticNet': ElasticNet(alpha=1.0, l1_ratio=0.5, random_state=42),
    'RandomForest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'GradientBoosting': GradientBoostingRegressor(n_estimators=100, random_state=42)
}

# Train and evaluate models
results = {}

for model_name, model in models.items():
    print(f"\nTraining {model_name}...")
    
    # Train
    model.fit(X_train_scaled, y_train)
    
    # Predict
    y_train_pred = model.predict(X_train_scaled)
    y_test_pred = model.predict(X_test_scaled)
    
    # Evaluate
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    
    results[model_name] = {
        'train_r2': train_r2,
        'test_r2': test_r2,
        'train_rmse': train_rmse,
        'test_rmse': test_rmse,
        'train_mae': train_mae,
        'test_mae': test_mae,
        'model': model,
        'predictions': y_test_pred
    }
    
    print(f"{model_name} - Train R²: {train_r2:.4f}, Test R²: {test_r2:.4f}")
    print(f"{model_name} - Train RMSE: {train_rmse:.4f}, Test RMSE: {test_rmse:.4f}")
    print(f"{model_name} - Train MAE: {train_mae:.4f}, Test MAE: {test_mae:.4f}")

In [ ]:
# Compare model performance
results_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Train R²': [results[m]['train_r2'] for m in results],
    'Test R²': [results[m]['test_r2'] for m in results],
    'Train RMSE': [results[m]['train_rmse'] for m in results],
    'Test RMSE': [results[m]['test_rmse'] for m in results],
    'Train MAE': [results[m]['train_mae'] for m in results],
    'Test MAE': [results[m]['test_mae'] for m in results]
})

print("\nModel Performance Comparison:")
print(results_df.to_string(index=False))

# Find best model
best_model_name = results_df.loc[results_df['Test R²'].idxmax(), 'Model']
print(f"\n✅ Best model based on Test R²: {best_model_name}")

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# R² comparison
results_df.plot(x='Model', y=['Train R²', 'Test R²'], kind='bar', ax=axes[0])
axes[0].set_title('R² Score Comparison')
axes[0].set_ylabel('R² Score')
axes[0].set_xlabel('Model')
axes[0].legend(loc='lower right')
axes[0].tick_params(axis='x', rotation=45)

# RMSE comparison
results_df.plot(x='Model', y=['Train RMSE', 'Test RMSE'], kind='bar', ax=axes[1])
axes[1].set_title('RMSE Comparison')
axes[1].set_ylabel('RMSE')
axes[1].set_xlabel('Model')
axes[1].legend(loc='upper right')
axes[1].tick_params(axis='x', rotation=45)

# MAE comparison
results_df.plot(x='Model', y=['Train MAE', 'Test MAE'], kind='bar', ax=axes[2])
axes[2].set_title('MAE Comparison')
axes[2].set_ylabel('MAE')
axes[2].set_xlabel('Model')
axes[2].legend(loc='upper right')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Plot actual vs predicted for best model
best_predictions = results[best_model_name]['predictions']

plt.figure(figsize=(10, 6))
plt.scatter(y_test, best_predictions, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Risk Score')
plt.ylabel('Predicted Risk Score')
plt.title(f'Actual vs Predicted Risk Score - {best_model_name}')
plt.tight_layout()
plt.show()

In [ ]:
# Residual plot for best model
residuals = y_test - best_predictions

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Residual plot
axes[0].scatter(best_predictions, residuals, alpha=0.5)
axes[0].axhline(y=0, color='r', linestyle='--', lw=2)
axes[0].set_xlabel('Predicted Risk Score')
axes[0].set_ylabel('Residuals')
axes[0].set_title(f'Residual Plot - {best_model_name}')

# Residual distribution
axes[1].hist(residuals, bins=30, edgecolor='black')
axes[1].set_xlabel('Residuals')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Residual Distribution')

plt.tight_layout()
plt.show()

print(f"Residual Statistics:")
print(f"Mean: {residuals.mean():.4f}")
print(f"Std: {residuals.std():.4f}")
print(f"Min: {residuals.min():.4f}")
print(f"Max: {residuals.max():.4f}")

## Summary

This notebook completed the following steps:
1. ✅ Loaded dataframe using df_loader
2. ✅ Applied SMOTE to balance LoanApproved data
3. ✅ Checked class distribution before and after SMOTE
4. ✅ Applied feature engineering module
5. ✅ Applied feature extraction (correlation, variance, importance-based selection)
6. ✅ Built regression pipeline with multiple models (Ridge, Lasso, ElasticNet, RandomForest, GradientBoosting)
7. ✅ Evaluated and compared model performance using R², RMSE, and MAE metrics